# Despliegue del modelo — Titanic

**Materia:** Analítica de Datos — Universidad Pontificia Bolivariana
**Equipo:** Nicolás Castrillón, Mateo Roldán, Matías Arango

Este notebook documenta el proceso de despliegue del modelo final (Random Forest optimizado con GridSearchCV, entrenado en `modelos_mineria_datos.ipynb`) mediante una interfaz gráfica hecha con **Streamlit**.

Contenido:
1. Carga y verificación del modelo final guardado.
2. Prueba del modelo con datos de ejemplo (simulando lo que hace la app).
3. Explicación del código de la aplicación `app/despliegue.py`.
4. Instrucciones para ejecutar la aplicación.
5. Instrucciones para tomar la captura de pantalla que pide el profesor.

## 1. Carga del modelo final

El modelo se guardó en la Fase 4 del notebook de modelos, en `models/modelo_final.pkl`, usando `joblib`. Es un objeto `Pipeline` de scikit-learn que incluye **todo** el preprocesamiento (imputación de nulos, escalado de numéricas, codificación dummy de categóricas) más el `RandomForestClassifier` ya optimizado — por eso la app puede recibir directamente los datos "crudos" que ingresa el usuario, sin que este tenga que preocuparse por escalarlos o codificarlos.

In [1]:
import joblib
import pandas as pd

modelo = joblib.load('../models/modelo_final.pkl')
print(type(modelo))
print(modelo)

<class 'sklearn.pipeline.Pipeline'>
Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Age', 'Fare', 'SibSp',
                                                   'Parch']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncode

## 2. Prueba del modelo con datos de ejemplo

Antes de construir la interfaz, comprobamos que el modelo cargado predice correctamente con un par de pasajeros hipotéticos, para verificar que el pipeline funciona de punta a punta igual que en el notebook de entrenamiento.

In [2]:
pasajeros_ejemplo = pd.DataFrame([
    {"Pclass": 1, "Sex": "female", "Age": 29, "SibSp": 0, "Parch": 0, "Fare": 80.0, "Embarked": "S"},
    {"Pclass": 3, "Sex": "male",   "Age": 25, "SibSp": 0, "Parch": 0, "Fare": 8.0,  "Embarked": "S"},
    {"Pclass": 2, "Sex": "female", "Age": 4,  "SibSp": 1, "Parch": 1, "Fare": 26.0, "Embarked": "C"},
])

predicciones = modelo.predict(pasajeros_ejemplo)
probabilidades = modelo.predict_proba(pasajeros_ejemplo)[:, 1]

resultado = pasajeros_ejemplo.copy()
resultado["Predicción"] = ["Sobrevive" if p == 1 else "No sobrevive" for p in predicciones]
resultado["Probabilidad de sobrevivir"] = probabilidades.round(3)
resultado

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,Predicción,Probabilidad de sobrevivir
0,1,female,29,0,0,80.0,S,Sobrevive,0.979
1,3,male,25,0,0,8.0,S,No sobrevive,0.118
2,2,female,4,1,1,26.0,C,Sobrevive,0.912


Los tres casos tienen sentido frente a los patrones históricos conocidos del Titanic: la mujer de primera clase con tarifa alta obtiene la mayor probabilidad de sobrevivir, y el hombre de tercera clase con tarifa baja obtiene la menor. Esto confirma que el pipeline guardado reproduce el comportamiento del modelo entrenado.

## 3. La aplicación de despliegue (`app/despliegue.py`)

La aplicación está hecha con **Streamlit** y hace lo siguiente:

1. **Carga el modelo** (`joblib.load`) una sola vez, usando `st.cache_resource` para no releerlo en cada interacción.
2. **Muestra un formulario** (`st.form`) con un campo por cada variable predictora: clase, sexo, edad, hermanos/cónyuge a bordo, padres/hijos a bordo, tarifa y puerto de embarque. Los campos usan `selectbox` para las categóricas (con etiquetas en español, traducidas internamente a los valores que espera el modelo) y `number_input` con límites mínimos y máximos para las numéricas.
3. **Valida los datos**: antes de predecir, revisa que la edad esté en un rango razonable (0–100), que la tarifa no sea negativa y que el número de familiares no sea negativo; si algo falla, muestra el error y no llama al modelo.
4. **Arma un DataFrame de una fila** con exactamente los mismos nombres de columnas (`Pclass`, `Sex`, `Age`, `SibSp`, `Parch`, `Fare`, `Embarked`) que espera el `ColumnTransformer` del pipeline.
5. **Predice** con `modelo.predict()` y `modelo.predict_proba()`, y muestra el resultado con `st.success` / `st.error`, además de las probabilidades de cada clase.
6. **Maneja errores de ejecución** con un bloque `try/except` alrededor de la predicción, para que un problema inesperado no haga que la aplicación se caiga en la cara del usuario.
7. En la barra lateral, muestra información del modelo y un gráfico de importancia de variables, calculado directamente desde el `RandomForestClassifier` ya entrenado.

El código completo está en `app/despliegue.py`, dentro de la estructura del proyecto.

## 4. Cómo iniciar la aplicación

**Desde la terminal (recomendado), parado en la carpeta raíz del proyecto:**

```bash
pip install -r requirements.txt
streamlit run app/despliegue.py
```

Streamlit abrirá automáticamente una pestaña del navegador en `http://localhost:8501`. Si no se abre sola, copia esa dirección y pégala en el navegador.

**Desde Visual Studio Code:**

1. Abre la carpeta del proyecto en VS Code (`File > Open Folder`).
2. Abre una terminal integrada (`Terminal > New Terminal`).
3. Activa el entorno virtual si usas uno (`.venv\Scripts\activate` en Windows, o el que tengas configurado).
4. Ejecuta `pip install -r requirements.txt` (solo la primera vez, o si cambiaron las dependencias).
5. Ejecuta `streamlit run app/despliegue.py`.
6. VS Code puede preguntarte si quieres abrir el puerto en el navegador — acepta, o entra manualmente a `http://localhost:8501`.

Para detener la aplicación, vuelve a la terminal y presiona `Ctrl+C`.

## 5. Cómo tomar la captura de pantalla que pide el profesor

1. Inicia la aplicación con `streamlit run app/despliegue.py` y espera a que se abra en el navegador.
2. Llena el formulario con un pasajero de ejemplo (por ejemplo, el que usamos arriba: primera clase, mujer, 29 años, 0 familiares, tarifa 80, Southampton).
3. Haz clic en **"Predecir supervivencia"** y espera a que aparezca el resultado (la sección "Resultado" con la predicción y las probabilidades).
4. Toma la captura de pantalla completa de la ventana del navegador, de forma que se vea: el formulario con los datos ingresados, el resultado de la predicción y, si quieres, la barra lateral con la importancia de variables.
   - En Windows: `Win + Shift + S` para recortar solo la ventana, o `Alt + Impr Pant` para capturar la ventana activa completa.
5. Guarda la imagen (por ejemplo como `captura_despliegue.png`) y adjúntala en la tarea de Teams junto con los notebooks, tal como pide el profesor.